# 03 · Ознаки моделі сонця

Що з погоди і положення сонця пов'язане з генерацією, які ознаки дублюють одна одну і як їх розплутали. В кінці — набір для моделі **«сьогодні»** (скільки сонце дасть сьогодні). Таблиця — з `02_dataset`.

In [1]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import data_io as io, dataset
from labels import SETS, VARS, OWN, GROUP_ORDER, label, group
plt.rcParams.update({'figure.figsize': (11, 3.5), 'axes.grid': True, 'grid.alpha': .3})
pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', None)

df = pd.read_parquet(io.DATA / 'dataset.parquet')
use = df.enough & ~df.clipped
print(f'годин для навчання: {use.sum():,}')

годин для навчання: 5,907


## Які ознаки є

У навчанні буде лише те, що відомо **наперед**: прогнози, сонце, дата. Факт ERA5 — тільки для порівняння (скільки точності губить прогноз).

In [2]:
feats = [c for c in df.columns if c not in dataset.NOT_FEATURES]
f = pd.DataFrame({'група': [group(c) for c in feats], 'ознака': [label(c).split(': ')[-1] for c in feats], 'змінна в коді': feats})
f.sort_values('група', key=lambda g: g.map({v: i for i, v in enumerate(GROUP_ORDER)}), kind='stable').set_index(['група', 'ознака'])

змінна в коді
група                       ознака                                                               
сонце і дата (рахуємо самі) висота сонця                                            sun_elevation
                            напрямок сонця                                            sun_azimuth
                            день року (sin)                                               doy_sin
                            день року (cos)                                               doy_cos
                            снігопад за 3 доби                                         snow_3d_cm
факт (ERA5)                 радіація                                 archive__shortwave_radiation
                            пряма радіація                              archive__direct_radiation
                            розсіяна радіація                          archive__diffuse_radiation
                            хмарність                                        archive__cloud_cover
                            опади                                          archive__precipitation
                            снігопад                                            archive__snowfall
                            температура                                   archive__temperature_2m
                            вологість                               archive__relative_humidity_2m
                            вітер                                         archive__wind_speed_10m
                            сніг лежить                                       archive__snow_depth
                            індекс ясності                                           csi__archive
                            частка розсіяної                                       dfrac__archive
свіжий прогноз              радіація                                forecast__shortwave_radiation
                            пряма радіація                             forecast__direct_radiation
                            розсіяна радіація                         forecast__diffuse_radiation
                            хмарність                                       forecast__cloud_cover
                            опади                                         forecast__precipitation
                            снігопад                                           forecast__snowfall
                            температура                                  forecast__temperature_2m
                            вологість                              forecast__relative_humidity_2m
                            вітер                                        forecast__wind_speed_10m
                            індекс ясності                                          csi__forecast
                            частка розсіяної                                      dfrac__forecast
прогноз за добу             радіація                                    day1__shortwave_radiation
                            пряма радіація                                 day1__direct_radiation
                            розсіяна радіація                             day1__diffuse_radiation
                            хмарність                                           day1__cloud_cover
                            опади                                             day1__precipitation
                            снігопад                                               day1__snowfall
                            температура                                      day1__temperature_2m
                            вологість                                  day1__relative_humidity_2m
                            вітер                                            day1__wind_speed_10m
                            індекс ясності                                              csi__day1
                            частка розсіяної                                          dfrac__day1
прогноз за 2 доби           радіація                              

## Що найбільше пов'язане з генерацією

Кореляція з генерацією на чистих годинах (від −1 до 1; ближче до ±1 — сильніший зв'язок). Погода — по наборах: видно, скільки зв'язку губиться, коли прогноз дальший.

In [3]:
c = df[use].corr(numeric_only=True)['pv_w']
w = pd.DataFrame({SETS[k]: {**{VARS[v]: c.get(f'{k}__{v}') for v in VARS},
                            'індекс ясності': c.get(f'csi__{k}'), 'частка розсіяної': c.get(f'dfrac__{k}')} for k in SETS}).dropna(how='all')
w = w.reindex(w['факт (ERA5)'].abs().sort_values(ascending=False).index).round(2)
w.index.name = 'ознака'
w['змінна в коді'] = [next((k for k, v in VARS.items() if v == n), {'індекс ясності': 'csi', 'частка розсіяної': 'dfrac'}.get(n)) for n in w.index]
w

,факт (ERA5),свіжий прогноз,прогноз за добу,прогноз за 2 доби,змінна в коді
ознака,,,,,
радіація,0.91,0.91,0.90,0.88,shortwave_radiation
пряма радіація,0.88,0.85,0.84,0.83,direct_radiation
вологість,-0.64,-0.59,-0.58,-0.57,relative_humidity_2m
частка розсіяної,-0.61,-0.65,-0.64,-0.61,dfrac
температура,0.52,0.51,0.51,0.51,temperature_2m
розсіяна радіація,0.50,0.70,0.68,0.67,diffuse_radiation
хмарність,-0.30,-0.26,-0.26,-0.23,cloud_cover
індекс ясності,0.25,0.27,0.25,0.23,csi
сніг лежить,-0.20,NaN,NaN,NaN,snow_depth


In [4]:
o = pd.DataFrame({'ознака': list(OWN.values()), 'кореляція з генерацією': [round(c[k], 2) for k in OWN], 'змінна в коді': list(OWN)})
o.reindex(o['кореляція з генерацією'].abs().sort_values(ascending=False).index).set_index('ознака')

,кореляція з генерацією,змінна в коді
ознака,,
висота сонця,0.79,sun_elevation
день року (cos),-0.37,doy_cos
снігопад за 3 доби,-0.16,snow_3d_cm
напрямок сонця,-0.08,sun_azimuth
день року (sin),-0.06,doy_sin


## Ознаки, що дублюють одна одну

Декілька пар кажуть майже одне й те саме:

In [5]:
pair = ['sun_elevation', 'clearsky_ghi', 'forecast__shortwave_radiation', 'forecast__direct_radiation', 'day1__shortwave_radiation',
        'doy_cos', 'forecast__temperature_2m']
df[use][pair].rename(columns=label).corr().round(2)

,висота сонця,радіація ясного неба,свіжий прогноз: радіація,свіжий прогноз: пряма радіація,прогноз за добу: радіація,день року (cos),свіжий прогноз: температура
висота сонця,1.00,0.99,0.80,0.64,0.81,-0.37,0.42
радіація ясного неба,0.99,1.00,0.80,0.63,0.81,-0.31,0.37
свіжий прогноз: радіація,0.80,0.80,1.00,0.95,0.97,-0.35,0.47
свіжий прогноз: пряма радіація,0.64,0.63,0.95,1.00,0.90,-0.33,0.47
прогноз за добу: радіація,0.81,0.81,0.97,0.90,1.00,-0.35,0.46
день року (cos),-0.37,-0.31,-0.35,-0.33,-0.35,1.00,-0.80
свіжий прогноз: температура,0.42,0.37,0.47,0.47,0.46,-0.80,1.00


Як розплутали:

- **Висота сонця ↔ радіація ясного неба 0.99** (ясне небо рахується з висоти сонця). Замість ясного неба — **індекс ясності** (clear-sky index) = прогноз радіації / радіація ясного неба: 1 — ясно, 0.1 — суцільні хмари. Стандартний прийом у сонячних прогнозах: готова функція `pvlib.irradiance.clearsky_index`; [Lauret et al., Solar 2022](https://hal.univ-reunion.fr/hal-03964785v1) — більшість прогнозистів моделюють саме індекс, і нормування на ясне небо дає кращі прогнози незалежно від типу моделі.
- **Загальна ↔ пряма радіація 0.94** (загальна = пряма + розсіяна). Замість прямої і розсіяної — **частка розсіяної** = розсіяна / загальна: ~1 — димка чи хмари, ~0.2 — пряме сонце.
- **Свіжий ↔ за добу 0.96** (і так для всіх величин, 0.83–1.0). Сирі day1/day2 відновлюються зі свіжого + змін, тож беремо **свіжий + зміни** (day1 − day2, свіжий − day1, розкид).
- **День року ↔ температура −0.80** — пора року. Лишаємо обидві, але перевіримо в моделі з ними і без них: температура може нести своє (гарячі панелі дають менше).
- День року ↔ висота сонця лише −0.37: висота міняється протягом дня, день року — ні. Не дублікати.

In [6]:
pd.DataFrame({'висота сонця ↔ індекс ясності': [round(df[use].sun_elevation.corr(df[use].csi__forecast), 2)],
              'загальна радіація ↔ частка розсіяної': [round(df[use].forecast__shortwave_radiation.corr(df[use].dfrac__forecast), 2)]},
             index=['кореляція після розплутування'])

,висота сонця ↔ індекс ясності,загальна радіація ↔ частка розсіяної
кореляція після розплутування,-0.12,-0.78


## Набір ознак моделі «сьогодні»

Скільки сонце дасть сьогодні. Основа — свіжий прогноз (він є лише на найближчі години). Модель «завтра» (на прогнозі за добу) — окремо, пізніше.

In [7]:
today = dataset.features('today')
TRY = {'doy_sin', 'doy_cos', 'forecast__temperature_2m'}
pd.DataFrame({'ознака': [label(x) for x in today], 'змінна в коді': today,
              'перевірити з/без': ['так' if x in TRY else '' for x in today]}).set_index('ознака')

,змінна в коді,перевірити з/без
ознака,,
висота сонця,sun_elevation,
напрямок сонця,sun_azimuth,
снігопад за 3 доби,snow_3d_cm,
свіжий прогноз: радіація,forecast__shortwave_radiation,
свіжий прогноз: індекс ясності,csi__forecast,
свіжий прогноз: частка розсіяної,dfrac__forecast,
свіжий прогноз: хмарність,forecast__cloud_cover,
свіжий прогноз: вологість,forecast__relative_humidity_2m,
свіжий прогноз: вітер,forecast__wind_speed_10m,


Пари, що лишились з кореляцією ≥ 0.7:

In [8]:
cc = df[use][today].corr()
pairs = [(label(a), label(b), round(cc.loc[a, b], 2)) for i, a in enumerate(today) for b in today[i + 1:] if abs(cc.loc[a, b]) >= .7]
pd.DataFrame(pairs, columns=['ознака', 'ознака ', 'кореляція']).sort_values('кореляція', key=abs, ascending=False).reset_index(drop=True)

,ознака,ознака,кореляція
0,висота сонця,свіжий прогноз: радіація,0.80
1,день року (cos),свіжий прогноз: температура,-0.80
2,свіжий прогноз: радіація,свіжий прогноз: частка розсіяної,-0.78
3,свіжий прогноз: індекс ясності,свіжий прогноз: хмарність,-0.74
4,свіжий прогноз: індекс ясності,свіжий прогноз: частка розсіяної,-0.73
5,свіжий прогноз: частка розсіяної,свіжий прогноз: хмарність,0.73


Вище 0.9 нема. Залишились природні: висота сонця ↔ радіація (вище сонце — більше світла), індекс ясності ↔ хмарність ↔ частка розсіяної (три погляди на хмари), день року ↔ температура (перевіримо).

## Чи показує розкид прогнозів, наскільки прогноз помилиться

Розкид — наскільки три версії прогнозу (за 2 доби, за добу, свіжий) розходились на ту саму годину. Промах — наскільки свіжий прогноз радіації розійшовся з фактом ERA5.

Порівнюємо **у відсотках від ясного неба**: інакше полудень завищує і розкид, і промах просто тому, що там більше сонця.

In [9]:
u = df[use].copy()
LEVELS = ['майже однакові', 'мало', 'середньо', 'помітно', 'сильно']
u['miss'] = (u.archive__shortwave_radiation - u.forecast__shortwave_radiation).abs()
u['spread_rel'] = u.dyn__shortwave_radiation__spread / u.clearsky_ghi * 100
u['miss_rel'] = u.miss / u.clearsky_ghi * 100
u['версії розходились'] = pd.qcut(u.spread_rel, 5, labels=LEVELS)
t = u.groupby('версії розходились', observed=True)[['spread_rel', 'miss_rel']].median().round(1)
t.columns = ['розкид, % ясного неба', 'промах, % ясного неба']; t

,"розкид, % ясного неба","промах, % ясного неба"
версії розходились,,
майже однакові,0.8,6.3
мало,4.1,10.3
середньо,7.7,12.8
помітно,12.5,15.2
сильно,24.3,25.4


Те саме всередині годин зі схожим сонцем (за радіацією ясного неба), промах у Вт/м²:

In [10]:
rows = {}
for lo, hi, name in ((150, 350, 'слабке 150–350'), (350, 550, 'середнє 350–550'), (550, 900, 'сильне 550–900')):
    s = u[(u.clearsky_ghi >= lo) & (u.clearsky_ghi < hi)].copy()
    s['g'] = pd.qcut(s.dyn__shortwave_radiation__spread, 5, labels=LEVELS)
    rows[f'сонце {name} ({len(s)} год)'] = s.groupby('g', observed=True).miss.median().round(0).astype(int)
r = pd.DataFrame(rows).T; r.columns.name = 'версії розходились →'; r

версії розходились →,майже однакові,мало,середньо,помітно,сильно
сонце слабке 150–350 (1519 год),18,22,31,35,40
сонце середнє 350–550 (1089 год),23,42,48,63,64
сонце сильне 550–900 (1465 год),17,58,73,95,106


In [11]:
print(f"рангова кореляція розкид ↔ промах: {u.spread_rel.corr(u.miss_rel, method='spearman'):.2f}")

рангова кореляція розкид ↔ промах: 0.34


## Висновки

1. **Головна ознака — радіація**: кореляція з генерацією 0.91 з фактом і зі свіжим прогнозом, 0.90 за добу, 0.88 за 2 доби. Прогноз за 2 доби губить небагато. Далі — пряма радіація, висота сонця. Вологість пов'язана обернено (вологість ≈ хмари, туман).
2. **Дублікати розплутано.** Радіацію ясного неба замінили індексом ясності (з висотою сонця тепер −0.12); пряму/розсіяну — часткою розсіяної; сирі day1/day2 — змінами.
3. **Модель «сьогодні» — 20 ознак**, пар вище 0.9 нема. День року і температура — перевірено в `05`: температура потрібна, день року — ні.
4. **Розкид прогнозів працює, але помірно.** У % від ясного неба: коли версії майже однакові, промах ~6 %; коли сильно розходяться — ~25 % (у 4 рази). Всередині схожого сонця — у 2–6 разів, найсильніше в яскраві години. Рангова кореляція 0.34. Беремо як ознаку і для меж прогнозу («у гіршому разі»).